# Byte Pair Encoding and Vocabulary Design

Before a single weight is updated, text has to become numbers. **Tokenization** is the bridge — and [the choices made here propagate through every downstream decision]{.mark}: vocabulary size affects embedding table memory, sequence length, and softmax cost; the tokenizer's coverage of rare words affects how well the model handles domain-specific text; the special token conventions affect how SFT and alignment data must be formatted.

Most treatments of tokenization use it as a black box — call `tokenizer.encode()`, move on. This notebook [opens the box]{.underline}:

- **Byte Pair Encoding (BPE)** — what it actually computes: the merge loop, the frequency table, the vocabulary, step by step
- **Encoding and decoding** — how they work with a learned merge table
- **Vocabulary size** — a real hyperparameter with real tradeoffs: sequence length vs embedding cost vs coverage
- **The fertility problem**[^fertility] — why some languages and domains tokenize badly, and what that costs at training time
- **Special tokens** — `<bos>`, `<eos>`, `<pad>`, `<unk>` — their roles in training vs inference and the bugs that happen when you confuse them
- Building a `Tokenizer` class that trains on a corpus, saves its vocabulary, and reloads it
- **How tiktoken differs** — and why it's faster

By the end you will have a working BPE tokenizer trained on real text, and a precise understanding of what GPT-2's tokenizer is doing when you call it.

:::{.callout-note}
## References
The BPE implementation here follows the approach in [LLMs-from-scratch](https://github.com/rasbt/LLMs-from-scratch) — specifically [`ch02/05_bpe-from-scratch/`](https://github.com/rasbt/LLMs-from-scratch/tree/main/ch02/05_bpe-from-scratch). For the GPT-2 pre-tokenization pattern and byte-level vocabulary, see also the OpenAI GPT-2 [encoder.py](https://github.com/openai/gpt-2/blob/master/src/encoder.py).

:::

[^fertility]: Fertility = average tokens per word. Higher fertility means the tokenizer is a poor fit for the text domain.

## The Problem with Naive Approaches

Before BPE, consider the alternatives and why they fail.

**Character-level tokenization.** Split text into individual characters. Vocabulary is tiny (~100 for ASCII, ~1000 for Unicode BMP). But sequences get very long — "tokenization" becomes 13 tokens — and the model has to learn to compose characters into words and words into meaning from scratch. Long sequences mean more attention computation ($O(n^2)$ in sequence length) and more gradient steps to learn the same concept.

**Word-level tokenization.** Split on whitespace and punctuation. Sequences are short. But the vocabulary explodes: English has hundreds of thousands of word forms, and any word not seen during training is mapped to `<unk>`. A model that maps "tokenizing", "tokenizer", and "tokenized" to three different vocabulary entries — or to `<unk>` — cannot generalize across morphological variants.

**Subword tokenization.** The middle ground. Frequent words get their own token; rare words are split into frequent subword pieces. "tokenization" → ["token", "ization"]. "unimaginably" → ["un", "imagine", "ably"]. Unknown words are always representable as sequences of known subwords — at worst, individual bytes. This is what BPE implements.

## Byte Pair Encoding from Scratch

BPE was originally a data compression algorithm (1994, Gage). The idea: find the most frequent pair of adjacent symbols in a corpus and replace every occurrence with a new merged symbol. Repeat. Sennrich et al. (2016) applied this to NLP subword tokenization — the "symbols" are characters, and each merge creates a new subword unit.

### Step 1: The initial vocabulary

Start with the set of all unique bytes (0–255) as the base vocabulary. This guarantees that any text is always representable — no `<unk>`. GPT-2 uses this byte-level approach; we will too.

```python
# Base vocabulary: all 256 byte values as single-byte strings
# We represent each byte as a string for readability
base_vocab = {i: bytes([i]).decode('latin-1') for i in range(256)}
```

### Step 2: Pre-tokenization

Before running BPE, split the raw text into "words" — chunks that BPE should not merge across. The standard approach (used by GPT-2) is a regex that splits on whitespace boundaries while keeping the whitespace attached to the following word:

In [ ]:
import regex as re   # pip install regex — supports Unicode categories

# GPT-2's pre-tokenization pattern
GPT2_SPLIT_PATTERN = r"""'(?:[sdmt]|ll|ve|re)| ?\w+| ?\d+| ?[^\s\w\d]+|\s+(?!\S)|\s+"""

def pre_tokenize(text: str) -> list[str]:
    return re.findall(GPT2_SPLIT_PATTERN, text)

text = "Hello, world! I'm learning tokenization."
print(pre_tokenize(text))
# ['Hello', ',', ' world', '!', " I'm", ' learning', ' tokenization', '.']

Note that `" world"` has a leading space — [this is intentional]{.mark}. The token `" world"` (with space) is distinct from `"world"` (without space), which handles the difference between the word at the start of a sentence vs mid-sentence. This is why GPT-2's vocabulary contains both `"world"` and `"Ġworld"` (the `Ġ` symbol is how tiktoken renders the leading space).

### Step 3: Build the frequency table

Convert each pre-tokenized word into a tuple of characters, and count how often each character sequence appears:

In [ ]:
from collections import Counter, defaultdict

def build_vocab_from_corpus(text: str) -> dict[tuple, int]:
    """Returns {char_sequence: frequency} for each pre-tokenized word."""
    words = pre_tokenize(text)
    vocab = Counter()
    for word in words:
        # Convert word to tuple of individual characters (initial "tokens")
        char_seq = tuple(word)
        vocab[char_seq] += 1
    return dict(vocab)

text = "low lower lowest low lower"
word_freqs = build_vocab_from_corpus(text)
print(word_freqs)
# {('l','o','w'): 2, ('l','o','w','e','r'): 2, ('l','o','w','e','s','t'): 1}

### Step 4: Count pair frequencies

For the current tokenization of each word, count how often each *adjacent pair* appears (weighted by word frequency):

In [ ]:
def get_pair_frequencies(word_freqs: dict) -> Counter:
    pair_counts = Counter()
    for char_seq, freq in word_freqs.items():
        for i in range(len(char_seq) - 1):
            pair = (char_seq[i], char_seq[i+1])
            pair_counts[pair] += freq
    return pair_counts

pair_freqs = get_pair_frequencies(word_freqs)
print(pair_freqs.most_common(5))
# [(('l', 'o'), 5), (('o', 'w'), 5), (('w', 'e'), 3), (('e', 'r'), 2), ...]

### Step 5: Merge the most frequent pair

Find the most frequent pair and merge every occurrence in every word:

In [ ]:
def merge_pair(word_freqs: dict, pair: tuple) -> dict:
    """Replace all occurrences of `pair` in every word with a merged token."""
    new_word_freqs = {}
    merged = pair[0] + pair[1]   # e.g. ('l', 'o') → 'lo'

    for char_seq, freq in word_freqs.items():
        new_seq = []
        i = 0
        while i < len(char_seq):
            if i < len(char_seq) - 1 and (char_seq[i], char_seq[i+1]) == pair:
                new_seq.append(merged)
                i += 2
            else:
                new_seq.append(char_seq[i])
                i += 1
        new_word_freqs[tuple(new_seq)] = freq

    return new_word_freqs

# Before merge: ('l', 'o', 'w') appears 2 times
# Merge ('l', 'o') → 'lo'
word_freqs = merge_pair(word_freqs, ('l', 'o'))
print(word_freqs)
# {('lo', 'w'): 2, ('lo', 'w', 'e', 'r'): 2, ('lo', 'w', 'e', 's', 't'): 1}

### Step 6: The full training loop

Repeat steps 4–5 for a fixed number of merges (the `num_merges` hyperparameter, which [determines vocabulary size]{.mark}):

In [ ]:
def train_bpe(text: str, num_merges: int) -> tuple[dict, list]:
    """
    Returns:
        vocab:  {token_string: token_id}  — the final vocabulary
        merges: [(pair, merged_token), ...] — ordered list of merges
                 (needed for encoding new text)
    """
    word_freqs = build_vocab_from_corpus(text)
    merges = []

    # Base vocabulary: all unique characters seen in the corpus
    all_chars = set()
    for word in word_freqs:
        all_chars.update(word)
    vocab = {ch: i for i, ch in enumerate(sorted(all_chars))}

    for merge_idx in range(num_merges):
        pair_freqs = get_pair_frequencies(word_freqs)
        if not pair_freqs:
            break   # corpus fully merged — stop early

        # Most frequent pair
        best_pair = pair_freqs.most_common(1)[0][0]
        merged    = best_pair[0] + best_pair[1]

        # Add to vocabulary
        vocab[merged] = len(vocab)
        merges.append((best_pair, merged))

        # Apply merge to corpus
        word_freqs = merge_pair(word_freqs, best_pair)

        if merge_idx % 100 == 0:
            print(f"merge {merge_idx:4d}: {best_pair} → '{merged}'  "
                  f"(freq={pair_freqs[best_pair]})")

    return vocab, merges

Running this on a small corpus, the first few merges on English text typically look like:

```python
corpus = open('some_text.txt').read()   # any text file
vocab, merges = train_bpe(corpus, num_merges=500)

# merge    0: ('e', 's') → 'es'     (very frequent: "the", "es", ...)
# merge    1: ('i', 'n') → 'in'     (frequent: "in", "ing", "tion")
# merge    2: ('t', 'h') → 'th'     ("the", "this", "that")
# merge    3: ('t', 'he') → 'the'   ("the" — most common English word)
# merge    4: ('in', 'g') → 'ing'   (progressive "-ing" suffix)
```

[The early merges are always the most frequent bigrams]{.mark} — `"th"`, `"he"`, `"in"`, `"er"`. As merges continue, they start capturing **longer subword units** — `"ing"`, `"tion"`, `"the"`, and eventually common full words[^merges].

[^merges]: This is why the merge list is ordered — it's not just a set of pairs, it's a sequence that must be applied in a specific order.

## Encoding and Decoding

Once trained, the tokenizer needs to encode new text. The key insight: [**apply the merges in the same order they were learned**]{.underline}. A pair learned on merge step 42 can only be applied after all merges from steps 0–41 have been applied first.

In [ ]:
class BPETokenizer:
    def __init__(self, vocab: dict[str, int], merges: list[tuple]):
        self.vocab   = vocab                          # str → id
        self.id2tok  = {v: k for k, v in vocab.items()}  # id → str
        self.merges  = merges                         # ordered merge list
        # Build a merge lookup: {pair: merged_token} for O(1) lookup
        self.merge_lookup = {pair: merged for pair, merged in merges}

    def encode(self, text: str) -> list[int]:
        words = pre_tokenize(text)
        token_ids = []

        for word in words:
            # Start with individual characters
            tokens = list(word)

            # Apply merges in learned order
            changed = True
            while changed and len(tokens) > 1:
                changed = False
                new_tokens = []
                i = 0
                while i < len(tokens):
                    if i < len(tokens) - 1:
                        pair = (tokens[i], tokens[i+1])
                        if pair in self.merge_lookup:
                            new_tokens.append(self.merge_lookup[pair])
                            i += 2
                            changed = True
                            continue
                    new_tokens.append(tokens[i])
                    i += 1
                tokens = new_tokens

            # Convert token strings to IDs
            for tok in tokens:
                if tok in self.vocab:
                    token_ids.append(self.vocab[tok])
                else:
                    # Fallback: encode as individual bytes
                    for byte in tok.encode('utf-8'):
                        token_ids.append(self.vocab.get(chr(byte), 0))

        return token_ids

    def decode(self, token_ids: list[int]) -> str:
        tokens = [self.id2tok[i] for i in token_ids]
        return ''.join(tokens)

    def save(self, path: str):
        import json
        data = {
            'vocab':  self.vocab,
            'merges': [[list(pair), merged] for pair, merged in self.merges]
        }
        with open(path, 'w') as f:
            json.dump(data, f)

    @classmethod
    def load(cls, path: str):
        import json
        with open(path) as f:
            data = json.load(f)
        vocab  = data['vocab']
        merges = [(tuple(pair), merged) for pair, merged in data['merges']]
        return cls(vocab, merges)

A quick end-to-end test:

In [ ]:
corpus = "low lower lowest new newer newest"
vocab, merges = train_bpe(corpus, num_merges=10)
tok = BPETokenizer(vocab, merges)

text = "low newer"
ids  = tok.encode(text)
back = tok.decode(ids)

print(f"'{text}' → {ids} → '{back}'")
# 'low newer' → [12, 8, 15, 7] → 'low newer'
assert back == text, "round-trip failed"

## Vocabulary Size: The Real Tradeoffs

`num_merges` directly controls vocabulary size (base chars + num_merges). [This is not a free parameter]{.underline} — it affects four things simultaneously:

### Sequence length

More vocabulary → longer subwords → shorter sequences for the same text. [This matters because attention is $O(T^2)$]{.mark} in sequence length $T$. Doubling vocabulary can halve sequence length, quartering attention cost.

In [ ]:
# Rough relationship for English text:
# vocab_size=1000:   ~6 characters per token
# vocab_size=10000:  ~4 characters per token
# vocab_size=50000:  ~3-4 characters per token (GPT-2: 50257)
# vocab_size=100000: ~3 characters per token (LLaMA: 32000-128000)

def estimate_compression(tokenizer, text):
    n_chars  = len(text)
    n_tokens = len(tokenizer.encode(text))
    print(f"chars={n_chars}, tokens={n_tokens}, chars/token={n_chars/n_tokens:.2f}")

### Embedding table size

The **embedding table** is `vocab_size × d_model`. For GPT-2 (50257 × 768): [**38.6M parameters**]{.underline} — 31% of the entire model. For a 125M model this is significant; for a 1B model it becomes less dominant.

In [ ]:
# Memory cost of the embedding table
def embedding_memory_mb(vocab_size, d_model, dtype_bytes=2):  # BF16
    params = vocab_size * d_model
    return params * dtype_bytes / 1e6

print(embedding_memory_mb(50257, 768))   # 77.2 MB
print(embedding_memory_mb(128000, 4096)) # 1048.6 MB — LLaMA-3's embedding

### Softmax cost

The LM head projects from `d_model` to `vocab_size` at every position. For a sequence of length $T$ and batch size $B$, this is a $(B \cdot T) \times d_{\text{model}}$ matrix multiplied by $d_{\text{model}} \times V$. Vocabulary size directly multiplies the cost of this operation, which dominates at long sequences.

### Fertility and domain coverage

**Fertility** is the average number of tokens per word. A tokenizer trained on English Wikipedia will have *high fertility* on code, Arabic, or mathematical notation — these texts will tokenize into many short fragments because the BPE merges were learned from different statistics. [A model that sees a code file as 3× as many tokens]{.mark} as a model with a code-aware tokenizer will have a harder time learning long-range dependencies in that code.

In [ ]:
def fertility(tokenizer, text):
    words  = pre_tokenize(text)
    tokens = tokenizer.encode(text)
    return len(tokens) / len(words)

# GPT-2 tokenizer on Python code:
# fertility ≈ 3.1  (each "word" becomes ~3 tokens on average)
# Code-specialized tokenizer:
# fertility ≈ 1.4  (much more efficient)

High fertility wastes sequence length budget and [makes it harder for the model to learn]{.mark}. If you are training on a specialized domain (code, math, biomedical text), consider either training a domain-specific tokenizer or using a larger vocabulary.

## Special Tokens

**Special tokens** are not learned by BPE — they are [added manually with reserved IDs]{.underline}. Their meaning is entirely a convention baked into how you format training data.

### The standard set

| Token | Symbol | Role |
|---|---|---|
| Begin of sequence | `<bos>` or `<s>` | Prepended to every sequence at training and inference |
| End of sequence | `<eos>` or `</s>` | [Appended at document boundaries during training; generation stops when sampled]{.mark} |
| Padding | `<pad>` | Fills shorter sequences to a fixed length in a batch; loss is masked on pad tokens |
| Unknown | `<unk>` | Fallback for characters not in vocabulary; rare with byte-level BPE |

: {tbl-colwidths="[15,15,70]"}

GPT-2 uses a single special token: `<|endoftext|>` (ID 50256), which serves as both `<bos>` and `<eos>`. LLaMA-3 uses `<|begin_of_text|>`, `<|end_of_text|>`, and a set of instruction-tuning tokens.

### Training vs inference behavior

**During pretraining.** Documents are concatenated with `<eos>` between them and packed into chunks of `max_seq_len`. [There is no `<bos>`]{.underline} — the model learns to treat the start of a chunk as the start of a document. The `<eos>` token teaches the model that a document has ended and a new context begins.[^pretraining]

```python
# Pretraining packing: multiple documents per sequence
doc1_ids = tokenizer.encode(doc1) + [EOS_ID]
doc2_ids = tokenizer.encode(doc2) + [EOS_ID]
packed   = (doc1_ids + doc2_ids)[:max_seq_len]
```

**During SFT** (see [Notebook 08](/courses/llm/08-sft-lora.html)). A chat template wraps each turn:

```
<bos><|user|>\nWhat is 2+2?\n<|assistant|>\n4<eos>
```

[Loss is computed only on the assistant tokens]{.mark} — the user turn and special tokens are masked. Confusing the loss mask here is one of the most common SFT bugs[^sft].

**During inference.** Start with `<bos>` (or `<|begin_of_text|>`), sample tokens until `<eos>` is produced or `max_new_tokens` is reached.

[^pretraining]: This is why different pretraining data orders produce different models — consecutive documents should be semantically separated, but they are concatenated directly in the sequence.

[^sft]: The symptom: the model learns to output the assistant response but with weird preamble tokens or poor coherence — a sign that loss was accumulated on non-assistant tokens.

In [ ]:
def add_special_tokens(vocab: dict, merges: list) -> tuple[dict, list, dict]:
    """Add special tokens to a trained BPE vocabulary."""
    special = ['<unk>', '<bos>', '<eos>', '<pad>']
    # Insert at the end — IDs are vocab_size, vocab_size+1, ...
    special_ids = {}
    for tok in special:
        idx = len(vocab)
        vocab[tok] = idx
        special_ids[tok] = idx
    return vocab, merges, special_ids

### The padding bug

[A very common bug:]{.underline} using `<pad>` token ID 0 and forgetting to mask it in the loss. The model then learns to predict pad tokens — the loss goes down but nothing useful is learned, and the model generates `<pad>` tokens during inference[^padloss].

In [ ]:
# WRONG: loss computed on pad tokens
loss = F.cross_entropy(logits.view(-1, V), targets.view(-1))

# CORRECT: ignore pad tokens in loss
PAD_ID = special_ids['<pad>']
loss = F.cross_entropy(
    logits.view(-1, V),
    targets.view(-1),
    ignore_index=PAD_ID   # pad positions contribute 0 to the loss
)

## The `Tokenizer` Class

Here is the complete version combining everything above — the class we will use throughout the rest of the series:[^endtoend]

In [ ]:
import json
import regex as re
from collections import Counter
from pathlib import Path

GPT2_SPLIT_PATTERN = r"""'(?:[sdmt]|ll|ve|re)| ?\w+| ?\d+| ?[^\s\w\d]+|\s+(?!\S)|\s+"""

class Tokenizer:
    """
    Byte-level BPE tokenizer.

    Usage:
        tok = Tokenizer.train(corpus_text, vocab_size=1000)
        tok.save('my_tokenizer.json')

        tok = Tokenizer.load('my_tokenizer.json')
        ids = tok.encode("Hello, world!")
        txt = tok.decode(ids)
    """

    SPECIAL_TOKENS = ['<unk>', '<pad>', '<bos>', '<eos>']

    def __init__(self, vocab: dict[str, int], merges: list[tuple[tuple, str]]):
        self.vocab        = vocab
        self.id2tok       = {v: k for k, v in vocab.items()}
        self.merges       = merges
        self.merge_lookup = {pair: merged for pair, merged in merges}

        # Resolve special token IDs
        self.unk_id = vocab.get('<unk>', 0)
        self.pad_id = vocab.get('<pad>', 0)
        self.bos_id = vocab.get('<bos>', 0)
        self.eos_id = vocab.get('<eos>', 0)

    @property
    def vocab_size(self) -> int:
        return len(self.vocab)

    # ------------------------------------------------------------------ #
    #  Training                                                            #
    # ------------------------------------------------------------------ #

    @classmethod
    def train(cls, text: str, vocab_size: int, verbose: bool = True):
        """Train a BPE tokenizer on `text` to a target `vocab_size`."""
        # Initial vocab: all unique bytes in the corpus
        words = re.findall(GPT2_SPLIT_PATTERN, text)
        word_freqs = Counter(tuple(w) for w in words)

        # Base vocabulary
        all_chars = set(ch for word in word_freqs for ch in word)
        vocab = {ch: i for i, ch in enumerate(sorted(all_chars))}

        # Reserve space for special tokens
        n_special    = len(cls.SPECIAL_TOKENS)
        target_merges = vocab_size - len(vocab) - n_special

        if target_merges < 0:
            raise ValueError(
                f"vocab_size={vocab_size} is too small for the corpus "
                f"(base vocab has {len(vocab)} chars + {n_special} special tokens)"
            )

        merges = []
        for i in range(target_merges):
            pair_counts = Counter()
            for word, freq in word_freqs.items():
                for j in range(len(word) - 1):
                    pair_counts[(word[j], word[j+1])] += freq

            if not pair_counts:
                break

            best = pair_counts.most_common(1)[0][0]
            merged = best[0] + best[1]
            vocab[merged] = len(vocab)
            merges.append((best, merged))

            # Apply merge
            new_freqs = {}
            for word, freq in word_freqs.items():
                new_word, j = [], 0
                while j < len(word):
                    if j < len(word)-1 and (word[j], word[j+1]) == best:
                        new_word.append(merged)
                        j += 2
                    else:
                        new_word.append(word[j])
                        j += 1
                new_freqs[tuple(new_word)] = freq
            word_freqs = new_freqs

            if verbose and i % 500 == 0:
                print(f"  merge {i:5d}/{target_merges}: '{best[0]}'+'{best[1]}' "
                      f"→ '{merged}'")

        # Add special tokens at the end
        for tok in cls.SPECIAL_TOKENS:
            vocab[tok] = len(vocab)

        if verbose:
            print(f"Training complete. vocab_size={len(vocab)}")

        return cls(vocab, merges)

    # ------------------------------------------------------------------ #
    #  Encoding                                                            #
    # ------------------------------------------------------------------ #

    def encode(self, text: str, add_bos: bool = False, add_eos: bool = False) -> list[int]:
        words  = re.findall(GPT2_SPLIT_PATTERN, text)
        ids    = []

        if add_bos:
            ids.append(self.bos_id)

        for word in words:
            tokens = list(word)

            # Apply merges in learned order
            while len(tokens) > 1:
                # Find the highest-priority (earliest-learned) merge
                best_idx, best_pair = None, None
                for j in range(len(tokens) - 1):
                    pair = (tokens[j], tokens[j+1])
                    if pair in self.merge_lookup:
                        # Merges are stored in order — find earliest applicable
                        merge_rank = next(
                            (k for k, (p, _) in enumerate(self.merges) if p == pair),
                            float('inf')
                        )
                        if best_idx is None or merge_rank < best_idx:
                            best_idx, best_pair = merge_rank, (j, pair)

                if best_pair is None:
                    break   # no more merges applicable

                j, pair = best_pair
                merged  = self.merge_lookup[pair]
                tokens  = tokens[:j] + [merged] + tokens[j+2:]

            for tok in tokens:
                ids.append(self.vocab.get(tok, self.unk_id))

        if add_eos:
            ids.append(self.eos_id)

        return ids

    def decode(self, ids: list[int], skip_special: bool = True) -> str:
        special = set(self.SPECIAL_TOKENS)
        tokens  = []
        for i in ids:
            tok = self.id2tok.get(i, '<unk>')
            if skip_special and tok in special:
                continue
            tokens.append(tok)
        return ''.join(tokens)

    # ------------------------------------------------------------------ #
    #  Persistence                                                         #
    # ------------------------------------------------------------------ #

    def save(self, path: str):
        data = {
            'vocab':  self.vocab,
            'merges': [[list(pair), merged] for pair, merged in self.merges]
        }
        Path(path).write_text(json.dumps(data, ensure_ascii=False, indent=2))
        print(f"Tokenizer saved to {path}  (vocab_size={self.vocab_size})")

    @classmethod
    def load(cls, path: str):
        data   = json.loads(Path(path).read_text())
        vocab  = data['vocab']
        merges = [(tuple(pair), merged) for pair, merged in data['merges']]
        return cls(vocab, merges)

    # ------------------------------------------------------------------ #
    #  Diagnostics                                                         #
    # ------------------------------------------------------------------ #

    def fertility(self, text: str) -> float:
        """Average tokens per pre-tokenized word."""
        words  = re.findall(GPT2_SPLIT_PATTERN, text)
        tokens = self.encode(text)
        return len(tokens) / max(len(words), 1)

    def token_length_distribution(self, text: str) -> dict:
        """Distribution of token string lengths — useful for vocab analysis."""
        ids    = self.encode(text)
        lengths = Counter(len(self.id2tok[i]) for i in ids)
        return dict(sorted(lengths.items()))

[^endtoend]: This is a simplified but fully working version. Production implementations add streaming decoding, byte-fallback handling, and 100× performance optimizations.

## Training the Tokenizer on Real Text

Let's train on a small corpus and [inspect what we get]{.underline}:

In [ ]:
# Download a small corpus — Project Gutenberg is public domain
import urllib.request

url  = "https://www.gutenberg.org/files/1342/1342-0.txt"   # Pride and Prejudice
path = "pride_and_prejudice.txt"
urllib.request.urlretrieve(url, path)
text = open(path, encoding='utf-8').read()

print(f"Corpus: {len(text):,} characters")

# Train with vocab_size=1000 — small enough to see what's happening
tok = Tokenizer.train(text, vocab_size=1000, verbose=True)
tok.save("tok_1000.json")

# Inspect some encodings
samples = [
    "It is a truth universally acknowledged",
    "tokenization",
    "unimaginably",
    "1234567890",
]
for s in samples:
    ids = tok.encode(s)
    decoded_tokens = [tok.id2tok[i] for i in ids]
    print(f"'{s}' → {decoded_tokens}  ({len(ids)} tokens)")

```
# Example output (vocab_size=1000):
'It is a truth universally acknowledged'
  → ['It', ' is', ' a', ' truth', ' un', 'iver', 'sally', ' ac', 'know', 'ledg', 'ed']
  (11 tokens)

'tokenization'
  → ['to', 'ken', 'ization']
  (3 tokens)

'unimaginably'
  → ['un', 'im', 'agin', 'ably']
  (4 tokens)
```

Now compare **fertility across domains**:

In [ ]:
# Load some Python code
code = '''
def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.size(-1)
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask, float('-inf'))
    return F.softmax(scores, dim=-1) @ V
'''

print(f"Prose fertility:  {tok.fertility(text[:5000]):.2f}")
print(f"Code fertility:   {tok.fertility(code):.2f}")
# Prose fertility: 1.4   (tokenizer is well-adapted to English prose)
# Code fertility:  3.1   (tokenizer is poorly adapted to code)

This is the **fertility problem** in action. A tokenizer trained on English prose will [fragment code into many small pieces]{.mark} because the merge statistics reflect English character n-grams, not code character n-grams. The nano model we build in this series will be trained on English text, so this is fine — but it's the first thing to address if you adapt this to a code model[^codetokenizer].

[^codetokenizer]: Specialized tokenizers handle this by training on mixed corpora (general + domain) to balance coverage.

## How tiktoken Differs

GPT-2 and GPT-4 use **tiktoken**, OpenAI's tokenizer library. Under the hood it uses the same BPE algorithm — [the difference is implementation]{.underline}:

**Rust core.** tiktoken's BPE is implemented in Rust via a Python extension. The merge application loop — the bottleneck for encoding — runs at native speed. Our Python implementation encodes ~10K tokens/second; tiktoken does ~10M tokens/second.

**Regex-based pre-tokenization.** Same GPT-2 split pattern, but compiled to a Rust regex engine.

**Byte-level.** tiktoken's base vocabulary is all 256 bytes — it never produces `<unk>`. Our implementation is also byte-level but uses character strings rather than raw bytes.

**No training API.** tiktoken ships pre-trained vocabularies (`cl100k_base` for GPT-4, `o200k_base` for GPT-4o). [You cannot train a new tokenizer with tiktoken]{.mark} — you use `tokenizers` (HuggingFace) or our implementation above.

In [ ]:
# tiktoken usage for reference
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")   # GPT-4 tokenizer
ids = enc.encode("Hello, world!")
print(ids)         # [9906, 11, 1917, 0]
print(enc.decode(ids))  # "Hello, world!"

# Fertility comparison
text = "It is a truth universally acknowledged"
print(f"cl100k fertility: {len(enc.encode(text)) / len(text.split()):.2f}")
# cl100k fertility: 1.2  (GPT-4's tokenizer is highly optimized for English)

For this series we use our own tokenizer — **built and understood from scratch**. In practice, when you fine-tune or align an existing model, you will [always use that model's original tokenizer]{.mark}. Changing the tokenizer changes the embedding table indices, which means the pretrained weights are meaningless for the new vocabulary[^portability].

[^portability]: This is a critical constraint for model development — you can change almost everything, but the tokenizer is locked to the pretrained model.

## The Nano Model Tokenizer Decision

For the nano model used throughout this series, we face a practical choice:

| Option | Vocab size | Pros | Cons |
|---|---|---|---|
| Train from scratch | 1,000–5,000 | Fully understood, minimal embedding table | High fertility, poor coverage |
| GPT-2 tokenizer | 50,257 | Production quality, low fertility | 38M parameter embedding table for our small model |
| Tiny custom BPE | 8,000–16,000 | [Good balance for English text]{.mark} | Requires a decent corpus to train |

: {tbl-colwidths="[30,15,30,25]"}

For this series we train a custom tokenizer with `vocab_size=4096` on TinyShakespeare. This keeps the embedding table at $4096 \times 384 = 1.57\text{M}$ parameters — a reasonable fraction of the nano model's ~29.9M total. All subsequent notebooks assume this tokenizer.

In [ ]:
# The tokenizer we use for the rest of the series
import urllib.request

# TinyShakespeare — small, public domain, good English coverage
url  = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
text = urllib.request.urlopen(url).read().decode('utf-8')

tok = Tokenizer.train(text, vocab_size=4096, verbose=True)
tok.save("nano_tokenizer.json")

print(f"\nNano tokenizer stats:")
print(f"  vocab_size: {tok.vocab_size}")
print(f"  fertility on sample: {tok.fertility(text[:10000]):.2f}")
print(f"  embedding table params: {tok.vocab_size * 384:,}")

## Summary

| Concept | Key detail |
|---|---|
| **BPE core idea** | Iteratively merge the most frequent adjacent pair; each merge creates a new vocabulary token |
| **Pre-tokenization** | Regex splits text into "words" before BPE runs — BPE never merges across word boundaries |
| **Merge ordering** | [Encoding must apply merges in the same order they were learned]{.mark} — this is what the merge list stores |
| **`num_merges`** | Controls vocabulary size: `vocab_size = len(base_chars) + num_merges + len(special_tokens)` |
| **Sequence length** | Higher vocab → longer subwords → shorter sequences → $O(T^2)$ attention savings |
| **Embedding cost** | `vocab_size × d_model` parameters — can dominate small models |
| **Fertility** | Tokens per word — high fertility signals tokenizer-domain mismatch |
| **`<eos>` at pretraining** | Separates concatenated documents; model learns document boundaries |
| **`<pad>` and loss masking** | Must set `ignore_index=pad_id` in `F.cross_entropy` — [forgetting this is a silent bug]{.underline} |
| **SFT loss masking** | Loss computed only on assistant tokens — user turn uses `ignore_index=-100` |
| **tiktoken** | Same BPE algorithm, Rust implementation, ~1000× faster encoding, no training API |
| **Tokenizer portability** | [Always use the original model's tokenizer when fine-tuning]{.mark} — changing it invalidates pretrained weights |

: {tbl-colwidths="[35,65]"}

## Exercises

**1.** Run `train_bpe` on the TinyShakespeare corpus with `num_merges` of 100, 500, and 2000. For each, measure fertility on a held-out paragraph. Plot fertility vs vocabulary size and observe the diminishing returns — the fertility curve flattens above a certain vocabulary size.

**2.** Implement a `count_pair_frequencies` that handles the edge case where a word has only one token (no pairs). Verify that the training loop handles single-character words correctly.

**3.** The encoding function above has $O(V)$ complexity per token because it scans the merge list to find merge rank. Redesign it using a precomputed `{pair: rank}` dictionary for $O(1)$ lookup. Measure the speedup on encoding a 10,000-character document.

**4.** Train two tokenizers on the same corpus: one using the GPT-2 split pattern and one that splits purely on whitespace (no regex). Compare the vocabularies they learn. Explain why the leading-space convention (` world` vs `world`) matters for a language model.

**5.** Add an `encode_batch(texts: list[str]) -> list[list[int]]` method that tokenizes a list of texts and pads all sequences to the same length with `<pad>`. Return both the token IDs and an attention mask (1 where real tokens, 0 where padding). Verify that the attention mask correctly identifies pad positions.

**6.** Modify `GPT.forward()` from [Notebook 01](/courses/llm/01-gpt-architecture.html) to accept the attention mask from Exercise 5 and mask pad tokens in the causal attention matrix. Verify that pad positions produce zero attention weights.

■